# Monthly Stock Performance Analysis of NVIDIA (NVDA) Using WRDS CRSP Data

## Analytical Problem
This project analyses NVIDIA's monthly stock performance since 2022 using WRDS CRSP data. It focuses on price trends, monthly returns, and volatility.

## Target Audience
Retail investors and beginner financial analysts.

## Data Source
WRDS - CRSP Monthly Stock File (crsp.msf) and CRSP Monthly Stock Header File (crsp.msfhdr)  
Accessed on: 26 April 2026

## Note
This notebook requires WRDS access to reproduce the data extraction step.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wrds

In [ ]:
db = wrds.Connection()

In [ ]:
start_date = '2022-01-01'
ticker = 'NVDA'

sql_query = f"""
SELECT
    b.htsymbol,
    a.date,
    ABS(a.prc) AS price,
    a.ret
FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol = '{ticker}'
AND a.date >= '{start_date}'
ORDER BY a.date
"""
print(sql_query)

In [ ]:
stock_joined = db.raw_sql(sql_query, date_cols=['date'])
stock_joined.head()

In [ ]:
stock_joined = stock_joined.dropna(subset=['date'])
stock_joined['ret'] = pd.to_numeric(stock_joined['ret'], errors='coerce')
stock_joined['price'] = pd.to_numeric(stock_joined['price'], errors='coerce')

stock_joined.head()
stock_joined.info()

In [ ]:
stock_joined = stock_joined.rename(columns={
    'htsymbol': 'ticker',
    'ret': 'monthly_return'
})
stock_joined.head()

In [ ]:
stock_joined['monthly_return'].describe()

In [ ]:
mean_return = stock_joined['monthly_return'].mean()
volatility = stock_joined['monthly_return'].std()

print("Average monthly return:", mean_return)
print("Monthly return volatility:", volatility)

In [ ]:
best_month = stock_joined.loc[stock_joined['monthly_return'].idxmax()]
worst_month = stock_joined.loc[stock_joined['monthly_return'].idxmin()]

print("Best month:")
print(best_month)

print("\nWorst month:")
print(worst_month)

In [ ]:
stock_by_return = (
    stock_joined[stock_joined['monthly_return'].notna()]
    .sort_values(by='monthly_return', ascending=False)
)

stock_by_return.head(10)

In [ ]:
import os
os.makedirs("images", exist_ok=True)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(stock_joined['date'], stock_joined['price'], marker='o')
plt.title('NVIDIA Monthly Stock Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.savefig('images/price_trend.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(stock_joined['date'], stock_joined['monthly_return'])
plt.title('NVIDIA Monthly Returns')
plt.xlabel('Date')
plt.ylabel('Monthly Return')
plt.axhline(0, color='black', linewidth=1)
plt.savefig('images/monthly_returns.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(stock_joined['monthly_return'].dropna(), bins=20, kde=True)
plt.title('Distribution of NVIDIA Monthly Returns')
plt.xlabel('Monthly Return')
plt.savefig('images/return_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Key Findings

 1. NVIDIA’s stock price generally increased over the sample period, showing strong market performance.
 2. Monthly returns were volatile, with some months generating very large positive gains and others producing losses.
 3. The descriptive statistics suggest that NVIDIA may offer attractive return potential, but investors should also be aware of the stock’s risk.

## Conclusion

 Using monthly CRSP data from WRDS, this project shows that NVIDIA delivered strong stock performance during the selected period, although return volatility remained high. For beginner investors and analysts, this suggests that NVIDIA may be attractive as a growth stock, but it also requires tolerance for risk.